In [2]:
import os
import re

import folium
import h3
import mcr_py.package.h3
import pandas as pd
import polars as pl

In [3]:
# parameters
city_name = "cologne"
mode = "nextbike"
bike_file = f"{mode}_availability_{city_name.lower()}.csv"
bike_zip_file = f"../data/sharing_locations_raw/{bike_file}.zip"
geometa_path = f"../data/stateful_variables/{city_name.lower()}_geometa.pkl"
sharing_positions_path = (
    f"../data/sharing_locations_clustered/{city_name.lower()}_{mode}"
)
# city_center = [52.5170365, 13.3888599]
city_center = [50.938361, 6.959974]

In [8]:
import zipfile

with zipfile.ZipFile(bike_zip_file).open(bike_file) as f:
    availabilities = pl.read_csv(f)

## Availabilities Data Preparation

In [9]:
availabilities = availabilities.rename(
    lambda name: name.replace(f"{mode}_availability_", "")
)

In [10]:
print(
    availabilities["valid_from"].min(),
    availabilities["valid_till"].max(),
)

2022-01-15 00:01:00 2024-04-30 23:56:00


In [17]:
availabilities = (
    availabilities.with_row_index("id")
    .with_columns(
        pl.col("geometry")
        .str.replace(r"[a-zA-Z\(]+", "")
        .str.replace(r"\)", "")
        .str.split(by=" ")
        .list.to_struct(fields=["lon", "lat"], n_field_strategy="max_width")
    )
    .unnest("geometry")
)

In [ ]:
availabilities.with_columns(
    availabilities["valid_from"].str.to_datetime(),
    availabilities["valid_till"].str.to_datetime(),
)

TypeError: DataFrame object does not support `Series` assignment by index

Use `DataFrame.with_columns`.

In [15]:
availabilities = mcr_py.package.h3.add_h3_cell_id_to_df_with_batching(
    availabilities, 8, n_batches=16 * 10
)

/workspaces/mcr-py/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


AssertionError: 

## Spatial and Temporal Discretization 

In [ ]:
def get_locations_at_time(df, time):
    return df[(df.valid_from <= time) & (df.valid_till >= time)]

In [ ]:
buckets = pd.date_range(
    availabilities.valid_from.min(), availabilities.valid_till.max(), freq="1H"
)
buckets

In [ ]:
import multiprocessing

from mcr_py.package import key
from tqdm.auto import tqdm

all_hex_ids = availabilities.h3_cell_id.unique()


def count_bikes_at_time(time_point):
    locations_at_time = get_locations_at_time(availabilities, time_point)

    n_bikes_per_hex = locations_at_time.groupby("h3_cell_id").size()
    n_bikes_per_hex = n_bikes_per_hex.reindex(all_hex_ids, fill_value=0)
    n_bikes_per_hex.name = time_point
    return n_bikes_per_hex


with multiprocessing.Pool(key.DEFAULT_N_PROCESSES) as pool:
    n_bikes_per_hex_per_time = list(
        tqdm(pool.imap(count_bikes_at_time, buckets), total=len(buckets))
    )

In [ ]:
n_bikes_per_hex_per_time = pd.DataFrame(n_bikes_per_hex_per_time)
n_bikes_per_hex_per_time.index.name = "time"

In [ ]:
n_bikes_per_hex_per_time.head()

In [ ]:
n_bikes_per_hex_per_time.to_csv(
    f"../data/sharing_locations_raw/{mode}_availability_{city_name.lower()}_bucketed.csv.zip"
)

In [ ]:
n_bikes_per_hex_per_time = pd.read_csv(
    f"../data/sharing_locations_raw/{mode}_availability_{city_name.lower()}_bucketed.csv.zip",
    index_col=0,
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_bikes = scaler.fit_transform(n_bikes_per_hex_per_time)

In [ ]:
from sklearn.metrics import calinski_harabasz_score
from sklearn_extra.cluster import KMedoids

stats = []
for k in range(2, 15):
    model = KMedoids(n_clusters=k, random_state=4711)
    pred_ = model.fit_predict(scaled_bikes)
    stats.append(
        {
            "k": k,
            "calinski_harabasz_score": calinski_harabasz_score(
                n_bikes_per_hex_per_time, pred_
            ),
        }
    )

In [ ]:
pd.DataFrame(stats).plot(x="k", y="calinski_harabasz_score")
# plt.savefig(f"../figures/sharing_clustering/{city_name}_{mode}_calinski_harabasz.png")

In [ ]:
# Number of clusters you want
n_clusters = 5

model = KMedoids(n_clusters=n_clusters, random_state=4711)
model.fit(scaled_bikes)

In [ ]:
pred = model.predict(scaled_bikes)

In [ ]:
pred

In [ ]:
n_bikes_per_hex_per_time["pred"] = pred

In [ ]:
visual = n_bikes_per_hex_per_time[
    n_bikes_per_hex_per_time.columns[n_bikes_per_hex_per_time.nunique() != 1]
]

In [ ]:
import seaborn as sns

sns.pairplot(visual, vars=visual.columns[:-1], hue="pred")

In [ ]:
medoids = n_bikes_per_hex_per_time.iloc[model.medoid_indices_]

In [ ]:
medoids

In [ ]:
from mcr_py.package.geometa import GeoMeta

geo_meta = GeoMeta.load(geometa_path)

In [ ]:
m = folium.Map(location=city_center, zoom_start=12)
geo_meta.add_to_folium_map(m)
h3.plot_h3_cells_on_folium(medoids.iloc[0].to_dict(), m, popup_callback=lambda x, y: x)
m

In [ ]:
time = medoids.index[3]
locations_at_time = get_locations_at_time(availabilities, time)
time

In [ ]:
medoids.index

In [ ]:
# center in cologne
m = folium.Map(location=city_center, zoom_start=13)

colors = ["red", "blue", "green", "yellow", "purple", "orange", "brown"]

for i, time in enumerate(medoids.index):
    locations_at_time = get_locations_at_time(availabilities, time)
    for point in locations_at_time[["lat", "lon"]].values:
        folium.CircleMarker(
            location=point, radius=2, color=colors[i], fill=True, fill_color="#000000"
        ).add_to(m)


m

In [ ]:
def derive_filename(s) -> str:
    s = re.sub(r"[^a-zA-Z0-9\-_.]", "_", str(s))
    s = s.replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    return s

In [ ]:
os.makedirs(sharing_positions_path, exist_ok=True)
for time in medoids.index:
    locations_at_time = get_locations_at_time(availabilities, time)
    filename = derive_filename(time) + ".csv"
    file_path = os.path.join(sharing_positions_path, filename)
    locations_at_time[["lat", "lon"]].to_csv(file_path, index=False)